## `Import Libraries`

This notebook only use regex and visual mapping. 

In [20]:
# importing libraries
import pandas as pd
import emoji
import re
import unicodedata
from unidecode import unidecode

## `Clean & Concat Dataset`

In [21]:
# set the path
path_to_df1 = "data/data_komentar_dengan_prediksi - data_komentar_dengan_prediksi(2).csv"
path_to_df2 = "data/youtube_chat_jogja_clean.csv"

# read the dataset using pandas dataframe
df1 = pd.read_csv(path_to_df1)
df2 = pd.read_csv(path_to_df2)

# display dataset heads
print(df1.columns)
print("\n")
print('-'*100)
print("\n")
print(df2.columns)

Index(['video_id', 'title', 'channel_name', 'tanggal', 'author', 'komentar',
       'label', 'komentar_clean', 'predicted_label'],
      dtype='str')


----------------------------------------------------------------------------------------------------


Index(['datetime', 'author_name', 'message', 'cleaned_message', 'label'], dtype='str')


In [22]:
# take only neccesary columns
df1_to_concat = df1[["komentar", "label"]]
df2_to_concat = df2[["message", "label"]]
df2_to_concat = df2_to_concat.rename(columns={
    "message" : "komentar",
})

In [23]:
# concat dataset
df_concat = pd.concat([df1_to_concat, df2_to_concat], axis=0, ignore_index=True)
df_concat.info()

<class 'pandas.DataFrame'>
RangeIndex: 16580 entries, 0 to 16579
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   komentar  16578 non-null  str  
 1   label     16580 non-null  int64
dtypes: int64(1), str(1)
memory usage: 259.2 KB


In [24]:
# check for missing and duplicated value
def check_missing_and_duplicate(dataframe):
    return print("Nan Values:\n", dataframe.isnull().values.sum(), "\nDuplicated Values:\n", dataframe.duplicated().values.sum())

check_missing_and_duplicate(df_concat)

Nan Values:
 2 
Duplicated Values:
 4850


In [25]:
def drop_missing_and_duplicate(dataframe):
    dataframe = dataframe.dropna()
    dataframe = dataframe.drop_duplicates().reset_index(drop=True)
    return dataframe

df_concat = drop_missing_and_duplicate(df_concat)

In [26]:
# there are text using emoji description, let's change them to emoji 
def convert_to_emoji(text):
    if isinstance(text, str):
        return emoji.emojize(text, language='alias')
    return text

df_concat["komentar"] = df_concat["komentar"].apply(convert_to_emoji)
print(df_concat.tail())

                                                komentar  label
11724                        bayar piro yo kui pesertane      0
11725                      mumet ndelokke dronne mas bro      0
11726                         Met ultah kotakuu,love you      0
11727  :hand-pink-waving: NATUNATOTO:hand-pink-waving...      0
11728                                  menyala jogjaku🔥🔥      0


In [27]:
df_concat.to_csv("data/data_concat.csv", index=False)

## `Preprocessing Text`

In [28]:
# check for imbalance class
df = pd.read_csv("data/data_concat.csv")
df_from_youtube = pd.read_csv("data/from-yt/only-comment/all_judol_ads.csv")
df["label"].value_counts()

label
0    8710
1    3019
Name: count, dtype: int64

In [29]:
df_from_youtube["label"].value_counts()

label
1    1058
Name: count, dtype: int64

In [30]:
df_final = pd.concat([df, df_from_youtube], axis=0, ignore_index=True)
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 12787 entries, 0 to 12786
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   komentar  12787 non-null  str  
 1   label     12787 non-null  int64
dtypes: int64(1), str(1)
memory usage: 199.9 KB


In [31]:
df_final["label"].value_counts()

label
0    8710
1    4077
Name: count, dtype: int64

In [32]:
# separate major and minor label
df_judi = df_final[df_final['label'] == 1]
df_bukan_judi = df_final[df_final['label'] == 0]

# undersampling major class using random sampling method
df_bukan_judi_undersampled = df_bukan_judi.sample(n=len(df_judi), random_state=42)

# put them back again
df_balanced = pd.concat([df_judi, df_bukan_judi_undersampled], axis=0)

# shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Verifikasi hasil
print(df_balanced['label'].value_counts())

# Simpan ke CSV untuk tahap modeling
df_balanced.to_csv("data/data_balanced.csv", index=False)

label
0    4077
1    4077
Name: count, dtype: int64


In [ ]:
def normalize_all_cases(text):
    if not isinstance(text, str): return ""
    
    # Visual Mapping
    visual_map = {
        'Δ': 'a', 'ά': 'a', 'ᗩ': 'a', '@': 'a',
        'ᗯ': 'w', 'vv': 'w',
        'σ': 'o', '𝐎': 'o', '0': 'o',
        '𝐍': 'n', 'И': 'n',
        '𝓽': 't', '𝓣': 't',
        '𝐒': 's', '🅢': 's',
    }
    
    for char, replacement in visual_map.items():
        text = text.replace(char, replacement)
    
    # Change symbol to ASCII
    text = unidecode(text)
    
    # Zalgo Normalization & Combining Marks (C̶ -> C)
    text = unicodedata.normalize('NFKD', text)
    text = "".join([c for c in text if unicodedata.category(c) != 'Mn'])
    
    # Lowercase & clean non-alfanumeric case
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Aggressive Space Collapse
    while True:
        new_text = re.sub(r'(?<=\b[a-z])\s+(?=[a-z]\b)', '', text)
        if new_text == text: break
        text = new_text
    
    return " ".join(text.split()).strip()

In [34]:
df_balanced['komentar_cleaned'] = df_balanced['komentar'].apply(normalize_all_cases)
df_balanced = df_balanced[df_balanced['komentar_cleaned'] != '']
df_balanced.info()


<class 'pandas.DataFrame'>
Index: 8153 entries, 0 to 8153
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   komentar          8153 non-null   str  
 1   label             8153 non-null   int64
 2   komentar_cleaned  8153 non-null   str  
dtypes: int64(1), str(2)
memory usage: 254.8 KB


In [35]:
df_balanced.to_csv("data/data_cleaned.csv", index=False)